# Rescaling Data for Comparison

In [ ]:
import os # Interoperable file paths
import pathlib # Find the home folder
import rioxarray as rxr # Work with geospatial raster data

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score

import rasterio
from scipy import ndimage

import xarray as xr
import numpy as np

import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
import matplotlib.pyplot as plt

## Load NEON hyperspectral images 
1. Load NEON .nc file 
2. Load EMIT .nc file.
3. Spectrally resample the NEON data to the EMIT bands using the function we built.
4. Spatially align and resample the two datasets to a common grid.
5. Compare the resulting resampled NEON and EMIT reflectance values for each band.

In [ ]:
# Define the base directory and file paths
base_dir_path = pathlib.Path(r"C:\Users\stem2\Documents\Capstone\AOP-EMIT\notebooks\exploratory\rn\data")

NEON_burned_file_path = base_dir_path / "NEON_D17_SOAP_DP3_298000_4100000_burned.nc"
NEON_unburned_file_path = base_dir_path / "NEON_D17_SOAP_DP3_298000_4101000_unburned.nc"
EMIT_burned_file_path = base_dir_path / "EMIT_L2A_RFL_20230731_SOAP_burned.nc"
EMIT_unburned_file_path = base_dir_path / "EMIT_L2A_RFL_20230731_SOAP_unburned.nc"

try:
    # Use the corrected xarray function for NetCDF files
    # 'data_array' is used since the file contains a single data variable
    NEON_burned = xr.open_dataarray(NEON_burned_file_path)
    NEON_unburned = xr.open_dataarray(NEON_unburned_file_path)
    EMIT_burned = xr.open_dataset(EMIT_burned_file_path)
    EMIT_unburned = xr.open_dataset(EMIT_unburned_file_path)

    print("Successfully loaded the NEON and EMIT burned and unburned files.")

    # You can now inspect the loaded data
    print(NEON_burned)
    print(NEON_unburned)
    print(EMIT_burned)
    print(EMIT_unburned)

except FileNotFoundError as e:
    print(f"Error: The file was not found. Please check the path.")
    print(e)
except Exception as e:
    print(f"An error occurred while loading the files: {e}")

# Spectral Resampling/Band Alignment 
Find Overlapping Bands: Identify the spectral bands in NEON that correspond to EMIT's bands.

Aggregate NEON Bands: If an EMIT band covers a range of NEON bands, you can average or sum the NEON bands within that range to create a comparable band for NEON.

Resampling Libraries: While rasterio focuses on spatial resampling, for spectral resampling (if not a direct band-to-band match), you'll primarily be working with the spectral dimension of your xarray.DataArray or NumPy array. You might write custom functions or use interpolation.

In [ ]:
import pathlib
import xarray as xr
import rioxarray as rxr
import numpy as np
import os
import h5py

# --- 1. Define the HDF5 to Xarray conversion function (your code) ---
def aop_h5refl2xarray(h5_filename):
    """
    Reads a NEON AOP reflectance HDF5 file and returns an xarray.Dataset with reflectance and weather quality indicator data.
    """
    import h5py
    import numpy as np
    import xarray as xr
    with h5py.File(h5_filename) as hdf5_file:
        print('Reading in ', h5_filename)
        sitename = list(hdf5_file.keys())[0]
        h5_refl_group = hdf5_file[sitename]['Reflectance']
        refl_dataset = h5_refl_group['Reflectance_Data']
        refl_array = refl_dataset[()].astype('float32')
        refl_arrayT = np.transpose(refl_array, (1, 0, 2))
        refl_arrayT = refl_array[::-1, :, :]
        refl_shape = refl_arrayT.shape
        wavelengths = h5_refl_group['Metadata']['Spectral_Data']['Wavelength'][:]
        fwhm = h5_refl_group['Metadata']['Spectral_Data']['FWHM'][:]
        wqi_array = h5_refl_group['Metadata']['Ancillary_Imagery']['Weather_Quality_Indicator'][()]
        wqi_arrayT = np.transpose(wqi_array, (1, 0))
        wqi_arrayT = wqi_array[::-1, :]
        metadata = {}
        metadata['shape'] = refl_shape
        metadata['no_data_value'] = float(refl_dataset.attrs['Data_Ignore_Value'])
        metadata['scale_factor'] = float(refl_dataset.attrs['Scale_Factor'])
        metadata['bad_band_window1'] = h5_refl_group.attrs['Band_Window_1_Nanometers']
        metadata['bad_band_window2'] = h5_refl_group.attrs['Band_Window_2_Nanometers']
        metadata['projection'] = h5_refl_group['Metadata']['Coordinate_System']['Proj4'][()].decode('utf-8')
        metadata['spatial_ref'] = h5_refl_group['Metadata']['Coordinate_System']['Coordinate_System_String'][()].decode('utf-8')
        metadata['EPSG'] = int(h5_refl_group['Metadata']['Coordinate_System']['EPSG Code'][()])
        map_info = str(h5_refl_group['Metadata']['Coordinate_System']['Map_Info'][()]).split(",")
        pixel_width = float(map_info[5])
        pixel_height = float(map_info[6])
        x_min = float(map_info[3]); x_min = int(x_min)
        y_max = float(map_info[4]); y_max = int(y_max)
        x_max = x_min + (refl_shape[1]*pixel_width); x_max = int(x_max)
        y_min = y_max - (refl_shape[0]*pixel_height); y_min = int(y_min)
        x_coords = np.linspace(x_min, x_max, num=refl_shape[1]).astype(float)
        y_coordsT = np.linspace(y_min, y_max, num=refl_shape[0]).astype(float)
        good_wavelengths = np.ones_like(wavelengths)
        for bad_window in [metadata['bad_band_window1'], metadata['bad_band_window2']]:
            bad_indices = np.where((wavelengths >= bad_window[0]) & (wavelengths <= bad_window[1]))[0]
            good_wavelengths[bad_indices] = 0
        good_wavelengths[-10:] = 0
        refl_xrT = xr.DataArray(
            refl_arrayT,
            dims=["y", "x", "wavelengths"],
            name="reflectance",
            coords={
                "y": ("y", y_coordsT),
                "x": ("x", x_coords),
                "wavelengths": ("wavelengths", wavelengths),
                "fwhm": ("wavelengths", fwhm),
                "good_wavelengths": ("wavelengths", good_wavelengths)
            }
        )
        wqi_xrT = xr.DataArray(
            wqi_arrayT,
            dims=["y", "x"],
            name="weather_quality_indicator",
            coords={
                "y": ("y", y_coordsT),
                "x": ("x", x_coords)
            }
        )
        dsT = xr.Dataset({
            "reflectance": refl_xrT,
            "weather_quality_indicator": wqi_xrT
        })
        for key, value in metadata.items():
            if key not in ['shape', 'extent', 'ext_dict']:
                dsT.attrs[key] = value
        return dsT

# --- 2. Define the new Spectral Resampling Function ---
def resample_neon_to_emit(neon_dataset, emit_bands_info):
    """
    Resamples NEON hyperspectral data to match EMIT's spectral bands.
    This version is compatible with the new aop_h5refl2xarray Dataset output.
    """
    if 'reflectance' not in neon_dataset.data_vars:
        raise ValueError("Input xarray.Dataset does not contain a 'reflectance' variable.")
    
    neon_reflectance = neon_dataset['reflectance']
    
    resampled_bands = []
    resampled_wavelengths = []
    
    neon_wavelengths = neon_reflectance['wavelengths'].values
    
    for band_info in emit_bands_info:
        center = band_info['center']
        fwhm = band_info['fwhm']
        
        min_wl = center - fwhm / 2
        max_wl = center + fwhm / 2
        
        matching_indices = (neon_wavelengths >= min_wl) & (neon_wavelengths <= max_wl)
        
        if np.sum(matching_indices) > 0:
            matching_neon_bands = neon_reflectance.isel(wavelengths=matching_indices)
            neon_aggregated_band = matching_neon_bands.mean(dim="wavelengths", keep_attrs=True)
            
            resampled_bands.append(neon_aggregated_band)
            resampled_wavelengths.append(center)
        else:
            print(f"Warning: No NEON bands found for EMIT band centered at {center:.2f} nm. Skipping.")

    if resampled_bands:
        resampled_da = xr.concat(resampled_bands, dim="band")
        resampled_da = resampled_da.assign_coords(wavelength=("band", resampled_wavelengths))
        resampled_da.attrs['long_name'] = "NEON data spectrally resampled to EMIT bands"
        return resampled_da
    else:
        return None

# --- 3. Full Processing Workflow ---
try:
    base_dir_path = pathlib.Path(r"C:\Users\stem2\Documents\Capstone\AOP-EMIT\notebooks\exploratory\rn\data")

    NEON_burned_h5 = base_dir_path / "NEON_D17_SOAP_DP3_298000_4100000_burned.h5"
    NEON_unburned_h5 = base_dir_path / "NEON_D17_SOAP_DP3_298000_4101000_unburned.h5"

    EMIT_burned_file_path = base_dir_path / "EMIT_L2A_RFL_20230731_SOAP_burned.nc"
    EMIT_unburned_file_path = base_dir_path / "EMIT_L2A_RFL_20230731_SOAP_unburned.nc"

    NEON_burned_ds = aop_h5refl2xarray(NEON_burned_h5)
    NEON_unburned_ds = aop_h5refl2xarray(NEON_unburned_h5)
    
    EMIT_burned_ds = xr.open_dataset(EMIT_burned_file_path)
    EMIT_unburned_ds = xr.open_dataset(EMIT_unburned_file_path)
    
    EMIT_burned = EMIT_burned_ds['reflectance']
    EMIT_unburned = EMIT_unburned_ds['reflectance']

    print("Successfully loaded all datasets.")

    # --- 4. Extract EMIT Band Metadata for Spectral Resampling ---
    emit_band_info = []
    for wl, fwhm in zip(EMIT_burned['wavelengths'].values, EMIT_burned['fwhm'].values):
        emit_band_info.append({"center": wl, "fwhm": fwhm})
        
    print("Successfully extracted EMIT band metadata.")

    # --- 5. SPECTRAL Resampling (NEON to EMIT) ---
    print("\nStarting spectral resampling on NEON burned data...")
    neon_burned_resampled = resample_neon_to_emit(NEON_burned_ds, emit_band_info)
    print("Spectral resampling for NEON burned data complete.")
    
    print("\nStarting spectral resampling on NEON unburned data...")
    neon_unburned_resampled = resample_neon_to_emit(NEON_unburned_ds, emit_band_info)
    print("Spectral resampling for NEON unburned data complete.")
    
    if neon_burned_resampled is None or neon_unburned_resampled is None:
        raise ValueError("Spectral resampling failed to return a valid dataset.")

    print("All spectral resampling complete.")

    # --- 6. SPATIAL Alignment (NEON to EMIT) ---
    print("\nStarting spatial alignment...")

    # Get the CRS from the NEON and EMIT datasets
    neon_crs_string = f"EPSG:{NEON_burned_ds.attrs['EPSG']}"
    emit_crs = EMIT_burned_ds.attrs['spatial_ref']

    # Assign the CRS to the spectrally resampled NEON data
    neon_burned_resampled = neon_burned_resampled.rio.write_crs(neon_crs_string)
    neon_unburned_resampled = neon_unburned_resampled.rio.write_crs(neon_crs_string)
    
    # Assign the CRS to the EMIT data as well, so reproject_match can find it
    EMIT_burned = EMIT_burned.rio.write_crs(emit_crs)
    EMIT_unburned = EMIT_unburned.rio.write_crs(emit_crs)

    # Reproject the spectrally resampled NEON data to match the EMIT grid
    neon_burned_aligned = neon_burned_resampled.rio.reproject_match(EMIT_burned)
    neon_unburned_aligned = neon_unburned_resampled.rio.reproject_match(EMIT_unburned)

    print("Spatial alignment complete.")

    # --- 7. Final Dimension Alignment and Subtraction ---
    print("\n--- Final Dimension Alignment and Calculating Difference Maps ---")

    # Rename EMIT dimensions to match NEON
    emit_burned_renamed = EMIT_burned.rename({
        'latitude': 'y',
        'longitude': 'x',
        'wavelengths': 'band'
    })
    emit_unburned_renamed = EMIT_unburned.rename({
        'latitude': 'y',
        'longitude': 'x',
        'wavelengths': 'band'
    })

    # Reorder EMIT dimensions to match NEON (band, y, x)
    emit_burned_aligned = emit_burned_renamed.transpose('band', 'y', 'x')
    emit_unburned_aligned = emit_unburned_renamed.transpose('band', 'y', 'x')

    # Now, subtract the maps.
    burned_difference = emit_burned_aligned.fillna(0) - neon_burned_aligned.fillna(0)
    unburned_difference = emit_unburned_aligned.fillna(0) - neon_unburned_aligned.fillna(0)

    print("Difference maps calculated successfully.")
    
    # --- 8. Print Final Results ---
    print("\n--- Final NEON Burned Aligned Data Info ---")
    print(neon_burned_aligned)
    print("\n--- Final EMIT Burned Data Info (for comparison) ---")
    print(EMIT_burned)
    print("\n--- Final NEON Unburned Aligned Data Info ---")
    print(neon_unburned_aligned)
    print("\n--- Final EMIT Unburned Data Info (for comparison) ---")
    print(EMIT_unburned)

    print("\n--- Burned Area Difference Map ---")
    print(burned_difference)

    print("\n--- Unburned Area Difference Map ---")
    print(unburned_difference)

except FileNotFoundError as e:
    print(f"Error: The file was not found. Please check the path.")
    print(e)
except Exception as e:
    print(f"An error occurred while loading or processing the files: {e}")

In [ ]:
import matplotlib.pyplot as plt

# Plot the burned area difference for a single band (e.g., band 100)
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

burned_difference.isel(band=100).plot.imshow(ax=ax[0], cmap='RdBu', center=0,
                                                cbar_kwargs={'label': 'Reflectance Difference (EMIT - NEON)'})
ax[0].set_title('Burned Area Difference (Band 100)')
ax[0].set_xlabel('Longitude')
ax[0].set_ylabel('Latitude')

# Plot the unburned area difference for the same band
unburned_difference.isel(band=100).plot.imshow(ax=ax[1], cmap='RdBu', center=0,
                                                  cbar_kwargs={'label': 'Reflectance Difference (EMIT - NEON)'})
ax[1].set_title('Unburned Area Difference (Band 100)')
ax[1].set_xlabel('Longitude')
ax[1].set_ylabel('Latitude')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pathlib

# Create a plots directory if it doesn't exist
plots_dir = pathlib.Path("./plots")
plots_dir.mkdir(exist_ok=True)

# Plot the burned area difference for a single band (e.g., band 100)
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

burned_difference.isel(band=100).plot.imshow(ax=ax[0], cmap='RdBu', center=0,
                                                cbar_kwargs={'label': 'Reflectance Difference (EMIT - NEON)'})
ax[0].set_title('Burned Area Difference (Band 100)')
ax[0].set_xlabel('Longitude')
ax[0].set_ylabel('Latitude')

# Plot the unburned area difference for the same band
unburned_difference.isel(band=100).plot.imshow(ax=ax[1], cmap='RdBu', center=0,
                                                  cbar_kwargs={'label': 'Reflectance Difference (EMIT - NEON)'})
ax[1].set_title('Unburned Area Difference (Band 100)')
ax[1].set_xlabel('Longitude')
ax[1].set_ylabel('Latitude')

plt.tight_layout()

# Save the figure to the plots directory
plt.savefig(plots_dir / 'difference_map_band_100.png', dpi=300, bbox_inches='tight')

# Display the plot in the notebook
plt.show()

In [ ]:

# --- 1. Load your spatially matched and CWC-calculated data ---
# This is a conceptual representation.
# You'd load your actual NEON/EMIT CWC and other spectral data here.
# Assuming you have a way to extract pixel values and align them.

# Example: Create a dummy DataFrame (replace with your actual data loading)
# Each row is a pixel, columns are features and the target.
num_pixels = 1000
data = {
    'NEON_CWC': np.random.rand(num_pixels) * 0.5 + 0.1, # Simulated CWC values
    'EMIT_CWC': np.random.rand(num_pixels) * 0.6 + 0.05,
    'NEON_Band_500nm': np.random.rand(num_pixels),
    'EMIT_Band_550nm': np.random.rand(num_pixels),
    # Add other bands/indices as features
    # 'Vegetation_Type': np.random.randint(0, 3, num_pixels) # Example target for classification
    'Drought_Stress_Level': np.random.randint(0, 3, num_pixels) # Example target for classification
}
df = pd.DataFrame(data)

# --- 2. Define Features (X) and Target (y) ---
# X will be your independent variables (inputs to the model)
# y will be your dependent variable (what you want to predict)

features = ['NEON_CWC', 'EMIT_CWC', 'NEON_Band_500nm', 'EMIT_Band_550nm']
X = df[features]
y = df['Drought_Stress_Level'] # Or 'Vegetation_Type', etc.

# --- 3. Split Data into Training and Testing Sets ---
# This is crucial for evaluating model performance on unseen data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y # stratify is good for imbalanced classes
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# --- 4. Fit the Decision Tree Model ---
# Initialize the classifier (or regressor)
clf = DecisionTreeClassifier(max_depth=5, random_state=42) # Limit depth to prevent overfitting

# Fit the model to the training data
clf.fit(X_train, y_train)

print("Model fitting complete!")

# --- 5. Evaluate the Model (using cross_val_score for robustness) ---
# Cross-validation provides a more reliable estimate of model performance
# by training and testing on different subsets of the data multiple times.
cv_scores = cross_val_score(clf, X, y, cv=5) # 5-fold cross-validation
print(f"\nCross-validation scores (accuracy): {cv_scores}")
print(f"Mean cross-validation accuracy: {np.mean(cv_scores):.2f}")

# You can also evaluate on the test set directly (after initial fitting)
from sklearn.metrics import accuracy_score, classification_report
y_pred = clf.predict(X_test)
print(f"\nAccuracy on test set: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report on test set:")
print(classification_report(y_test, y_pred))


# --- 6. Visualize the Decision Tree ---
# This helps interpret the rules learned by the tree.
plt.figure(figsize=(20, 10))
plot_tree(
    clf,
    feature_names=features,
    class_names=[str(c) for c in clf.classes_], # Convert class names to strings if needed
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree for CWC Comparison")
plt.show()

# --- 7. Interpret the Tree / Feature Importance ---
# You can see which features were most influential in the decision-making process.
feature_importances = pd.DataFrame({'feature': features, 'importance': clf.feature_importances_})
feature_importances = feature_importances.sort_values('importance', ascending=False)
print("\nFeature Importances:")
print(feature_importances)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pathlib

# Define the wavelength you want to visualize
green_wavelength_nm = 550

# Get the array of all wavelengths from the difference map
wavelengths_array = burned_difference['wavelength'].values

# Find the index of the wavelength that is closest to 550 nm
closest_band_index = np.argmin(np.abs(wavelengths_array - green_wavelength_nm))

# Get the actual wavelength of the band that was selected
actual_wavelength = wavelengths_array[closest_band_index]
print(f"Plotting difference map for green band at index {closest_band_index}, wavelength: {actual_wavelength:.2f} nm")

# Select the band using its integer index with .isel()
green_band_diff = burned_difference.isel(band=closest_band_index)

# Plot the difference map for the selected green band
fig, ax = plt.subplots(figsize=(8, 6))

green_band_diff.plot.imshow(ax=ax, cmap='RdBu', center=0,
                                cbar_kwargs={'label': 'Reflectance Difference (EMIT - NEON)'})
ax.set_title(f'Burned Area Difference (Green Band: {actual_wavelength:.2f} nm)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.tight_layout()

# Save the plot
plots_dir = pathlib.Path("./plots")
plots_dir.mkdir(exist_ok=True)
plt.savefig(plots_dir / f'burned_difference_green_band_{actual_wavelength:.2f}nm.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pathlib

# Define the wavelength you want to visualize
green_wavelength_nm = 550

# Get the array of all wavelengths from the difference map
wavelengths_array = unburned_difference['wavelength'].values

# Find the index of the wavelength that is closest to 550 nm
closest_band_index = np.argmin(np.abs(wavelengths_array - green_wavelength_nm))

# Get the actual wavelength of the band that was selected
actual_wavelength = wavelengths_array[closest_band_index]
print(f"Plotting difference map for green band at index {closest_band_index}, wavelength: {actual_wavelength:.2f} nm")

# Select the band using its integer index with .isel()
green_band_diff = unburned_difference.isel(band=closest_band_index)

# Plot the difference map for the selected green band
fig, ax = plt.subplots(figsize=(8, 6))

green_band_diff.plot.imshow(ax=ax, cmap='RdBu', center=0,
                                cbar_kwargs={'label': 'Reflectance Difference (EMIT - NEON)'})
ax.set_title(f'Unburned Area Difference (Green Band: {actual_wavelength:.2f} nm)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.tight_layout()

# Save the plot
plots_dir = pathlib.Path("./plots")
plots_dir.mkdir(exist_ok=True)
plt.savefig(plots_dir / f'unburned_difference_green_band_{actual_wavelength:.2f}nm.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import numpy as np
from scipy import stats # You will need to install scipy: 'pip install scipy'

# --- Define the bands for NDWI ---
nir_wavelength = 860   # Near-Infrared
swir_wavelength = 1240 # Short-wave Infrared

# Get the array of all wavelengths from the EMIT data
wavelengths_array = EMIT_unburned['wavelengths'].values

# Find the indices of the closest NIR and SWIR bands
nir_index = np.argmin(np.abs(wavelengths_array - nir_wavelength))
swir_index = np.argmin(np.abs(wavelengths_array - swir_wavelength))

print(f"Using NIR band at {wavelengths_array[nir_index]:.2f} nm and SWIR band at {wavelengths_array[swir_index]:.2f} nm")

# --- Calculate NDWI for EMIT and NEON ---
# Select the NIR and SWIR bands for both datasets
emit_nir = EMIT_unburned.isel(wavelengths=nir_index)
emit_swir = EMIT_unburned.isel(wavelengths=swir_index)

neon_nir = neon_unburned_aligned.isel(band=nir_index)
neon_swir = neon_unburned_aligned.isel(band=swir_index)

# Calculate NDWI using the formula
cwc_emit = (emit_nir - emit_swir) / (emit_nir + emit_swir)
cwc_neon = (neon_nir - neon_swir) / (neon_nir + neon_swir)

print("\nCWC (NDWI) maps calculated successfully.")

# --- Prepare data for statistical test ---
# Flatten the 2D data arrays into 1D arrays for comparison
cwc_emit_flat = cwc_emit.values.ravel()
cwc_neon_flat = cwc_neon.values.ravel()

# Filter out NaN values from both arrays consistently
valid_indices = ~np.isnan(cwc_emit_flat) & ~np.isnan(cwc_neon_flat)
cwc_emit_clean = cwc_emit_flat[valid_indices]
cwc_neon_clean = cwc_neon_flat[valid_indices]

print(f"Comparing {len(cwc_emit_clean)} valid pixels.")

# --- Perform Paired T-Test ---
# This part of the code requires the 'scipy' library.
# If you haven't installed it, run: 'pip install scipy' in your terminal or conda environment.
try:
    # A paired t-test compares the means of two related samples.
    t_statistic, p_value = stats.ttest_rel(cwc_emit_clean, cwc_neon_clean)
    
    mean_difference = np.mean(cwc_emit_clean - cwc_neon_clean)

    print("\n--- Paired T-Test Results for CWC Agreement ---")
    print(f"Mean Difference (EMIT - NEON): {mean_difference:.4f}")
    print(f"T-statistic: {t_statistic:.4f}")
    print(f"P-value: {p_value:.4f}")
    
    # Interpret the p-value
    alpha = 0.05
    if p_value < alpha:
        print("\nResult: There is a statistically significant difference between EMIT and NEON CWC values.")
    else:
        print("\nResult: There is no statistically significant difference between EMIT and NEON CWC values.")
        
except NameError:
    print("\nError: The 'stats' library (from scipy) is required for the t-test.")
    print("Please install it by running 'pip install scipy' in your environment.")

# Optional: Plot the CWC difference map
cwc_difference = cwc_emit - cwc_neon
plt.figure(figsize=(8, 6))
cwc_difference.plot(cmap='RdBu', center=0)
plt.title("CWC (NDWI) Difference Map (EMIT - NEON)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

Defining the Target Variable (y): This is the most critical difference.
the decision tree in a more exploratory way to understand splits? If you are just comparing NEON CWC to EMIT CWC, a decision tree might not be the most direct approach unless you categorize the difference between them.
Overfitting: Decision trees can easily overfit, especially with high-dimensional hyperspectral data.

max_depth: Limit the maximum depth of the tree (clf = DecisionTreeClassifier(max_depth=5)).

min_samples_leaf: Set a minimum number of samples required to be at a leaf node.

ccp_alpha: Use cost-complexity pruning (as shown in some scikit-learn examples) to find an optimal pruning parameter.

Cross-Validation (cross_val_score): Always use cross-validation to get a more robust estimate of your model's performance and to help with hyperparameter tuning.

Interpreting the Tree: The plot_tree function is invaluable for understanding the rules the model learned. Look at which features are used at the top of the tree (these are the most important splits) and what thresholds are being applied.